# SmartCart: Part 2: User Based Collaborative Filtering

##  Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import defaultdict

pd.set_option('display.max_columns', None)
np.random.seed(42)

## 2.1  Load Data & Build User Item Matrix

In [ ]:
user_data    = pd.read_csv('ecommerce_user_data.csv')
product_data = pd.read_csv('product_details.csv')

user_data['Timestamp'] = pd.to_datetime(user_data['Timestamp'])
# Remove duplicate (UserID, ProductID) pairs — keep last interaction
user_data = user_data.drop_duplicates(subset=['UserID', 'ProductID'], keep='last')

# Build user–item matrix; NaN = no interaction
user_item_matrix = user_data.pivot_table(
    index='UserID', columns='ProductID', values='Rating'
)
# Fill unrated entries with 0 for cosine similarity computation
user_item_matrix_filled = user_item_matrix.fillna(0)

n_users, n_products = user_item_matrix_filled.shape
n_rated   = (user_item_matrix_filled != 0).sum().sum()
sparsity  = 1 - n_rated / (n_users * n_products)

print(f"Users    : {n_users}")
print(f"Products : {n_products}")
print(f"Ratings  : {n_rated}")
print(f"Sparsity : {sparsity*100:.1f}%")
user_item_matrix_filled.head()

## 2.2  Cosine Similarity Between Users

For two users **u** and **v** with rating vectors **r_u** and **r_v**:

$$\text{sim}(u, v) = \frac{\mathbf{r}_u \cdot \mathbf{r}_v}{\|\mathbf{r}_u\| \cdot \|\mathbf{r}_v\|}$$

Values range from 0 (orthogonal / no overlap) to 1 (identical taste).

In [ ]:
sim_matrix = cosine_similarity(user_item_matrix_filled)
similarity_df = pd.DataFrame(
    sim_matrix,
    index=user_item_matrix_filled.index,
    columns=user_item_matrix_filled.index
)

print("Similarity matrix shape:", similarity_df.shape)
print("\nTop-5 most similar users to U000:")
print(similarity_df['U000'].drop('U000').sort_values(ascending=False).head())

## 2.3  Recommendation Function

**Algorithm:**
1. Find the `top_n_users` most similar users to the target user.
2. For each product the target user has **not** rated, compute a weighted predicted score:
$$\hat{r}_{u,p} = \frac{\sum_{v \in N(u)} \text{sim}(u,v) \cdot r_{v,p}}{\sum_{v \in N(u)} \text{sim}(u,v)}$$
3. Return the top-N products by predicted score.

In [ ]:
def get_recommendations(target_user, matrix, sim_df,
                        top_n_users=5, top_n_products=10):
    """
    Return a ranked list of top_n_products product IDs for target_user.
    Uses weighted average of similar users' ratings.
    Products already rated by target_user are excluded.
    """
    # Neighbours sorted by similarity (exclude self)
    neighbours = (
        sim_df[target_user]
        .drop(target_user)
        .sort_values(ascending=False)
        .head(top_n_users)
    )

    # Products already interacted with
    already_rated = set(
        matrix.columns[matrix.loc[target_user] > 0]
    )

    score_num = defaultdict(float)   # weighted rating sum
    score_den = defaultdict(float)   # similarity weight sum

    for neighbour, sim_weight in neighbours.items():
        if sim_weight <= 0:
            continue
        for product in matrix.columns:
            if product in already_rated:
                continue
            rating = matrix.loc[neighbour, product]
            if rating > 0:
                score_num[product] += sim_weight * rating
                score_den[product] += sim_weight

    # Normalise
    predicted = {
        p: score_num[p] / score_den[p]
        for p in score_num if score_den[p] > 0
    }

    ranked = sorted(predicted.items(), key=lambda x: x[1], reverse=True)
    return [p for p, _ in ranked[:top_n_products]]


# Quick demo
sample_recs = get_recommendations('U000', user_item_matrix_filled, similarity_df,
                                  top_n_users=5, top_n_products=10)
print("Top-10 recommendations for U000:", sample_recs)

## 2.4  Train / Test Split

For every user we randomly hold out **20 %** of their rated products as the test set.  
The model is trained on the remaining 80 % and evaluated against the hidden items.

In [ ]:
def train_test_split_users(matrix, test_ratio=0.2, random_state=42):
    """
    Hold out test_ratio of each user's rated products.
    Returns a zeroed-out train matrix and a dict {user: [held-out products]}.
    """
    rng = np.random.default_rng(random_state)
    train = matrix.copy()
    test_data = {}

    for user in matrix.index:
        rated = matrix.columns[matrix.loc[user] > 0].tolist()
        n_test = max(1, int(len(rated) * test_ratio))
        held_out = rng.choice(rated, size=n_test, replace=False).tolist()
        test_data[user] = held_out
        train.loc[user, held_out] = 0.0   # hide from model

    return train, test_data


train_matrix, test_data = train_test_split_users(user_item_matrix_filled)

total_test = sum(len(v) for v in test_data.values())
print(f"Total held-out interactions : {total_test}")
print(f"Avg held-out per user       : {total_test/len(test_data):.1f}")

## 2.5  Evaluation Metrics

| Metric | Formula | Interpretation |
|---|---|---|
| **Precision@K** | (hits in top-K) / K | How many of the K recommendations are relevant |
| **Recall@K** | (hits in top-K) / (total relevant) | How many relevant items were found |
| **MAP** | mean of AP across users | Rewards correct recommendations ranked higher |

**Relevant** = products in the user's held out test set.

In [ ]:
def precision_at_k(recommended, relevant, k):
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / k


def recall_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    hits = len(set(recommended[:k]) & set(relevant))
    return hits / len(relevant)


def average_precision(recommended, relevant, k):
    """AP for one user: average precision at each hit position up to K."""
    if not relevant:
        return 0.0
    hits, cumsum = 0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in set(relevant):
            hits += 1
            cumsum += hits / (i + 1)
    return cumsum / min(len(relevant), k)


def evaluate(train_mat, test_dict, k=10, top_n_users=5):
    """
    Rebuild similarity on train_mat, generate recs, compute metrics.
    Returns dict with Precision@K, Recall@K, MAP.
    """
    # Recompute similarity on training data only
    train_sim = cosine_similarity(train_mat)
    train_sim_df = pd.DataFrame(
        train_sim,
        index=train_mat.index,
        columns=train_mat.index
    )

    prec_list, rec_list, ap_list = [], [], []

    for user in train_mat.index:
        relevant = test_dict.get(user, [])
        if not relevant:
            continue
        recs = get_recommendations(user, train_mat, train_sim_df,
                                   top_n_users=top_n_users,
                                   top_n_products=k)
        prec_list.append(precision_at_k(recs, relevant, k))
        rec_list.append(recall_at_k(recs, relevant, k))
        ap_list.append(average_precision(recs, relevant, k))

    return {
        f'Precision@{k}': round(np.mean(prec_list), 4),
        f'Recall@{k}'   : round(np.mean(rec_list),  4),
        'MAP'           : round(np.mean(ap_list),    4),
    }


results_k10 = evaluate(train_matrix, test_data, k=10, top_n_users=5)
results_k5  = evaluate(train_matrix, test_data, k=5,  top_n_users=5)

print("=== Evaluation Results ===")
print(f"K = 5  : {results_k5}")
print(f"K = 10 : {results_k10}")

## 2.6  Coverage & Intra-List Diversity

- **Catalog Coverage**: fraction of all products that appear in at least one recommendation list.
- **Intra-List Diversity (ILD)**: average pairwise dissimilarity (1 − item cosine similarity) within each user's recommendation list, then averaged across users. Higher = more varied recommendations.

In [ ]:
def catalog_coverage(matrix, sim_df, k=10, top_n_users=5):
    """Fraction of catalogue that is recommended to at least one user."""
    all_products = set(matrix.columns)
    recommended  = set()
    for user in matrix.index:
        recs = get_recommendations(user, matrix, sim_df,
                                   top_n_users=top_n_users,
                                   top_n_products=k)
        recommended.update(recs)
    return round(len(recommended) / len(all_products), 4)


def intra_list_diversity(matrix, sim_df, k=10, top_n_users=5):
    """
    Average pairwise item dissimilarity within recommendation lists.
    Item similarity is computed via cosine similarity on the item–user matrix.
    """
    item_sim = cosine_similarity(matrix.T)           # products × products
    item_sim_df = pd.DataFrame(item_sim,
                               index=matrix.columns,
                               columns=matrix.columns)
    diversities = []
    for user in matrix.index:
        recs = get_recommendations(user, matrix, sim_df,
                                   top_n_users=top_n_users,
                                   top_n_products=k)
        if len(recs) < 2:
            continue
        pairs = [
            item_sim_df.loc[recs[i], recs[j]]
            for i in range(len(recs))
            for j in range(i + 1, len(recs))
        ]
        diversities.append(1 - np.mean(pairs))
    return round(np.mean(diversities), 4)


coverage  = catalog_coverage(user_item_matrix_filled, similarity_df, k=10)
diversity = intra_list_diversity(user_item_matrix_filled, similarity_df, k=10)

print(f"Catalog Coverage  : {coverage*100:.1f}% of products recommended")
print(f"Intra-List Diversity : {diversity:.4f}  (0=identical items, 1=maximally diverse)")

## 2.7 — Top-5 Recommendations per User (Sample)

We generate top-5 product recommendations for every user and enrich them with product names and categories.

In [ ]:
product_lookup = product_data.set_index('ProductID')[['ProductName', 'Category']]

all_recs = []
for user in user_item_matrix_filled.index:
    recs = get_recommendations(user, user_item_matrix_filled, similarity_df,
                               top_n_users=5, top_n_products=5)
    for rank, pid in enumerate(recs, start=1):
        row = {'UserID': user, 'Rank': rank, 'ProductID': pid}
        row.update(product_lookup.loc[pid].to_dict() if pid in product_lookup.index else {})
        all_recs.append(row)

recs_df = pd.DataFrame(all_recs)
recs_df.to_csv('recommendations_top5.csv', index=False)

print("Recommendations saved to recommendations_top5.csv")
print(f"Total rows: {len(recs_df)}")
recs_df[recs_df['UserID'].isin(['U000', 'U001', 'U002'])]

## 2.8  Visualisations

### (a) User Similarity Heatmap

In [ ]:
plt.figure(figsize=(14, 11))
mask = np.eye(len(similarity_df), dtype=bool)   # hide diagonal (self-similarity = 1)
sns.heatmap(
    similarity_df,
    mask=mask,
    cmap='YlOrRd',
    vmin=0, vmax=1,
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'Cosine Similarity'},
    xticklabels=True,
    yticklabels=True
)
plt.title('User–User Cosine Similarity Heatmap', fontsize=14, pad=12)
plt.xticks(fontsize=7, rotation=90)
plt.yticks(fontsize=7, rotation=0)
plt.tight_layout()
plt.savefig('part2_similarity_heatmap.png', dpi=150)
plt.show()

### (b) Evaluation Metrics Bar Chart (K = 5 vs K = 10)

In [ ]:
metrics_labels = ['Precision', 'Recall', 'MAP']
vals_k5  = [results_k5['Precision@5'],  results_k5['Recall@5'],  results_k5['MAP']]
vals_k10 = [results_k10['Precision@10'], results_k10['Recall@10'], results_k10['MAP']]

x = np.arange(len(metrics_labels))
width = 0.32

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, vals_k5,  width, label='K = 5',  color='steelblue',   edgecolor='white')
bars2 = ax.bar(x + width/2, vals_k10, width, label='K = 10', color='coral',        edgecolor='white')

ax.set_ylabel('Score')
ax.set_title('Collaborative Filtering — Evaluation Metrics')
ax.set_xticks(x)
ax.set_xticklabels(metrics_labels, fontsize=12)
ax.set_ylim(0, max(vals_k5 + vals_k10) * 1.3)
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points', ha='center', fontsize=9)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('part2_evaluation_metrics.png', dpi=150)
plt.show()

### (c) Coverage & Diversity Summary

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Precision@5', 'Recall@5', 'MAP (K=5)',
               'Precision@10', 'Recall@10', 'MAP (K=10)',
               'Catalog Coverage', 'Intra-List Diversity'],
    'Value': [
        results_k5['Precision@5'],
        results_k5['Recall@5'],
        results_k5['MAP'],
        results_k10['Precision@10'],
        results_k10['Recall@10'],
        results_k10['MAP'],
        coverage,
        diversity
    ]
})

fig, ax = plt.subplots(figsize=(9, 4))
ax.axis('off')
tbl = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.4, 1.6)
plt.title('Part 2 — Full Evaluation Summary', fontsize=13, pad=16)
plt.tight_layout()
plt.savefig('part2_summary_table.png', dpi=150)
plt.show()

print(summary.to_string(index=False))



### Key Observations
- The **85.5% sparsity** limits precision — users share few common products to anchor similarity estimates.
- **Recall@10 > Recall@5** as expected: a wider list captures more held-out items.
- **MAP** is low in sparse settings because correct items rarely appear near rank 1.
- **Coverage** shows what fraction of the catalogue can be surfaced by the system.
- These limitations are worth discussing in the report (Conceptual Question 1).